## Advanced Imputation Techniques in Pandas & Scikit-Learn ##

### Initialization ###

In [1]:
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer # Enables IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import BayesianRidge

url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/employees_dataset_with_missing.csv"
df = pd.read_csv(url)

### Statistics ###

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              884 non-null    float64
 1   income           851 non-null    float64
 2   education_years  909 non-null    float64
 3   experience       873 non-null    float64
 4   credit_score     1000 non-null   float64
dtypes: float64(5)
memory usage: 39.2 KB


In [3]:
df.describe()

,age,income,education_years,experience,credit_score
count,884.000000,851.000000,909.000000,873.000000,1000.000000
mean,35.279903,51060.017327,14.026104,10.009906,645.072636
std,9.842874,14941.524310,2.941903,5.164959,99.238021
min,2.587327,5894.170480,4.941464,-4.647243,332.329619
25%,28.764400,40898.473174,12.057375,6.380048,581.739504
50%,35.276021,51017.832658,13.997874,10.150157,648.175801
75%,41.549126,61037.912624,15.998938,13.435329,713.912314
max,73.527315,97896.613518,25.778713,26.215465,961.291020


In [4]:
print(f"Shape: {df.shape}")

Shape: (1000, 5)


In [5]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing (%)": (df.isnull().sum() / len(df)) * 100
})

display(missing[missing["Missing Count"] > 0].sort_values("Missing Count", ascending=False))

,Missing Count,Missing (%)
income,149,14.9
experience,127,12.7
age,116,11.6
education_years,91,9.1


### Multiple Imputation by Chained Equations (MICE) ###

The iterative imputation method uses a variety of estimators like random forest, Bayesian ridge, etc. to impute missing values. By default, Bayesian ridge is commonly used as it deems missing values as parameters to be learnt. 

In [6]:
rf_iterative_imputer = IterativeImputer(
    estimator = RandomForestRegressor(
        n_estimators = 10,
        random_state = 42
    ),
    random_state = 42,
    max_iter = 5
)

df_rf_iterative = pd.DataFrame(rf_iterative_imputer.fit_transform(df), columns = df.columns, index = df.index)

df_rf_iterative.sample(n = 5)

C:\Users\Phuong Thao\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,age,income,education_years,experience,credit_score
323,55.923873,57297.524657,19.023477,11.289880,542.332087
543,35.502677,43143.556151,13.074552,4.613120,590.233232
928,50.655240,66514.394525,18.734354,4.103154,520.943858
878,34.835771,48187.393711,14.731598,2.336497,581.861976
291,30.069991,87394.992761,16.936574,-2.439043,606.969068


### K-Nearest Neighbor Imputation ###

Similar to K-Nearest Neighbor clustering algorithm, this approach imputes missing values by resorting to calculating and using similarity among samples to estimate missing values in a given instance. Weighted similarity and custom metrics can likewise be utilized.

In [7]:
knn_imputer = KNNImputer(n_neighbors = 5, weights = "distance") # 5 clusters with inversely proportional distance-based weights
df_knn = pd.DataFrame(
    knn_imputer.fit_transform(df),
    columns = df.columns,
    index = df.index
)

df_knn.sample(n = 5)

,age,income,education_years,experience,credit_score
381,29.006074,42681.328878,11.128547,15.952743,667.618982
910,31.922218,75987.724992,12.393995,2.923664,598.174736
768,28.730329,51848.072102,12.450711,13.323514,752.941310
835,37.342147,55422.552878,13.100485,8.012211,659.297925
250,22.391160,54963.203483,13.617353,13.016238,757.534340


In [8]:
# Alternate approach: weights='uniform' - all selected neighbors (ten in this case) 
# Have equal weight in contributing to the estimation of the missing value in every target instance to be treated.
knn_uniform_imputer = KNNImputer(n_neighbors = 10, weights = "uniform")
df_uniform_knn = pd.DataFrame(
    knn_uniform_imputer.fit_transform(df),
    columns = df.columns,
    index = df.index
)

df_uniform_knn.sample(n = 5)

,age,income,education_years,experience,credit_score
454,29.697424,58482.644684,16.046156,-4.647243,776.765151
745,27.742562,53575.534741,13.110178,9.092484,621.490574
257,27.923305,56285.285140,12.737715,6.438896,792.449582
408,36.202956,45159.802417,16.434190,8.887626,704.090852
892,45.666747,43454.211642,16.304622,6.000297,593.055023


### Imputation With Multiple Estimators (Ensemble) ###

Another approach is to build multiple imputing estimators of different types, each yielding different imputed values from the full dataset. By inspecting each version based on the naturality, we can decide which one can provide the most consistent imputations for specific context of data. 

In [9]:
imputers = {
    'bayesian_ridge': IterativeImputer(estimator=BayesianRidge(), random_state=42),
    'extra_trees': IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=10, random_state=42), random_state=42),
    'rf_regressor': IterativeImputer(estimator=RandomForestRegressor(n_estimators=10, random_state=42), random_state=42)
}

imputed_datasets = {}

for name, imputer in imputers.items():
    imputed_datasets[name] = pd.DataFrame(imputer.fit_transform(df), columns=df.columns, index=df.index)

print("Imputed Dataset Versions based on Different Estimators:")
for name, dataset in imputed_datasets.items():
    print(f"{name}: Mean income = ${dataset['income'].mean():.2f}")

C:\Users\Phuong Thao\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Imputed Dataset Versions based on Different Estimators:
bayesian_ridge: Mean income = $51056.16
extra_trees: Mean income = $50992.66
rf_regressor: Mean income = $50957.85


C:\Users\Phuong Thao\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


### 📊 Summary ###

| Method       | Best For                                       | Data Size               | Best to Avoid For                          | Computational Cost | Interpretability |
| ------------ | ---------------------------------------------- | ----------------------- | ------------------------------------------ | ------------------ | ---------------- |
| **MICE**     | Correlated variables, mixed data types         | Medium–Large (1K+ rows) | Independent variables; Very large datasets | Medium             | High             |
| **KNN**      | Numerical data, local patterns                 | Small–Medium (<5K rows) | Categorical data; Sparse datasets          | High               | Medium           |
| **Ensemble** | Critical decisions, uncertainty quantification | Any size                | Quick prototyping; Limited resources       | Very High          | Low              |
